# Forecasting comparison — SPTLS vs the right model in each regime

SPTLS is run **out of the box** (one call: no unit-root pre-testing, no
lag/difference/variant choice). Each scenario compares it against the model a
careful practitioner would build *for that regime*.

1. **Linear (stationary & cointegrated):** SPTLS *ties* a tuned VAR and a
   Johansen VECM — reproducing the error-correction without estimating a
   cointegrating rank.
2. **Nonlinear, one step:** `SPTLS(dictionary="quadratic")` recovers a map a
   linear model cannot represent at any lag.
3. **Deterministic chaos:** SPTLS forecasts to the system's intrinsic
   predictability (Lyapunov) horizon; a linear VAR fails *structurally from
   step 1*.

Read honestly: on linear data SPTLS does not beat a correctly-specified model —
its value there is no model selection + the white-box reading. The forecasting
wins are nonlinear (2, 3).

In [ ]:
import sys, os, warnings
warnings.filterwarnings("ignore")
sys.path.insert(0, os.path.abspath("../../lib"))
import numpy as np, matplotlib.pyplot as plt, gsd
np.seterr(all="ignore"); np.set_printoptions(precision=3, suppress=True)
%matplotlib inline

# sanity: confirm the quadratic dictionary is actually active. If this fails,
# your gsd copy is stale (pre-rename) or a cached .pyc is shadowing it:
# delete lib/gsd/__pycache__, make sure lib/gsd/forecasting.py is current, and
# restart the kernel.
_chk = gsd.SPTLS(dictionary="quadratic").fit(gsd.datasets.logistic(T=80))
assert _chk.rsquared_mean > 0.99, (
    "Loaded gsd does NOT have the working 'quadratic' dictionary "
    "(got R2=%.3f). Stale copy/cache — clear lib/gsd/__pycache__, update "
    "lib/gsd/forecasting.py, and restart the kernel." % _chk.rsquared_mean)
print("gsd quadratic dictionary: active (sanity R2 = %.3f)" % _chk.rsquared_mean)

def _ols(X, y): return np.linalg.lstsq(X, y, rcond=None)[0]
def aic_p(tr, pmax=6):
    best = None
    for p in range(1, pmax+1):
        r = gsd.OLS(p).fit(tr); a = sum(e.aic for e in r.equations)
        if best is None or a < best[0]: best = (a, p)
    return best[1]
def fc(model, tr, h):
    res = model.fit(tr); hist = list(tr); out = []; sc = np.std(tr, 0) + 1
    for _ in range(h):
        nx = np.atleast_1d(res.forecast_next(np.array(hist)))
        nx = np.clip(np.nan_to_num(nx), -1e6*sc, 1e6*sc); out.append(nx); hist.append(nx)
    return np.array(out)
def fc_diff(tr, h):                   # the common "difference to stationarize" VAR
    d = np.diff(tr, axis=0); res = gsd.OLS(aic_p(d)).fit(d); hist = list(d); lvl = tr[-1].copy(); out=[]
    for _ in range(h):
        dn = np.atleast_1d(res.forecast_next(np.array(hist))); hist.append(dn)
        lvl = np.clip(lvl+dn, -1e8, 1e8); out.append(lvl.copy())
    return np.array(out)
def vecm_johansen(tr, h, rank=1):     # rigorous VECM via the Johansen procedure (numpy-only)
    Y = np.asarray(tr, float); d = Y.shape[1]; dY = np.diff(Y, axis=0)
    Z0, LY, DZ = dY[1:], Y[1:-1], dY[:-1]
    W = np.column_stack([np.ones(len(Z0)), DZ])
    R0 = Z0 - W@_ols(W, Z0); R1 = LY - W@_ols(W, LY); T = len(R0)
    S00=R0.T@R0/T; S11=R1.T@R1/T; S01=R0.T@R1/T
    w, V = np.linalg.eig(np.linalg.inv(S11)@S01.T@np.linalg.inv(S00)@S01)
    beta = np.real(V[:, np.argsort(-w.real)[:rank]])      # cointegrating vector(s)
    X = np.column_stack([np.ones(len(Z0)), LY@beta, DZ])
    B = np.array([_ols(X, Z0[:, i]) for i in range(d)])
    lvl = Y[-1].copy(); dz = dY[-1].copy(); out = []
    for _ in range(h):
        dn = B@np.concatenate([[1.0], lvl@beta, dz]); lvl = np.clip(lvl+dn, -1e8, 1e8); dz = dn; out.append(lvl.copy())
    return np.array(out)
def nrmse(f, a): return np.mean(np.sqrt(np.mean(((f-a)/(np.std(a,0)+1e-9))**2, 0)))

## 1. Linear regimes — SPTLS ties the right model, no model selection

A stationary VAR(1) and a cointegrated I(1) pair (an error-correction system,
whose correct model is a VECM). Multi-step level forecasts; mean over seeds.

In [ ]:
def var1(T=400, seed=1):
    r = np.random.default_rng(seed); A = np.array([[0.5,0.2],[-0.1,0.4]]); Y=np.zeros((T,2))
    for t in range(1,T): Y[t] = A@Y[t-1] + 0.3*r.standard_normal(2)
    return Y
def coint(T=400, seed=1, k=0.1):
    r = np.random.default_rng(seed); tr=np.cumsum(0.4*r.standard_normal(T)); sp=np.zeros(T)
    for t in range(1,T): sp[t]=(1-k)*sp[t-1]+0.4*r.standard_normal()
    return np.column_stack([tr+0.5*sp, tr-0.5*sp])

h=20
sA=[nrmse(fc(gsd.OLS(aic_p(var1(seed=s)[:-h])),var1(seed=s)[:-h],h),var1(seed=s)[-h:]) for s in range(10)]
spA=[nrmse(fc(gsd.SPTLS(),var1(seed=s)[:-h],h),var1(seed=s)[-h:]) for s in range(10)]
print("stationary :  VAR-AIC %.2f   SPTLS %.2f   -> tie" % (np.mean(sA), np.mean(spA)))
rows={"VECM-Johansen":[],"VAR-AIC(levels)":[],"SPTLS oob":[]}
for s in range(12):
    Y=coint(seed=s); tr,act=Y[:-h],Y[-h:]
    rows["VECM-Johansen"].append(nrmse(vecm_johansen(tr,h),act))
    rows["VAR-AIC(levels)"].append(nrmse(fc(gsd.OLS(aic_p(tr)),tr,h),act))
    rows["SPTLS oob"].append(nrmse(fc(gsd.SPTLS(),tr,h),act))
print("cointegration (level forecast, median over 12 seeds):")
for k in rows: print(f"   {k:>16}: {np.median(rows[k]):.2f}")

The theoretically-correct model for a cointegrated system is a **VECM**. We
estimate one rigorously via the **Johansen** reduced-rank procedure (pure
numpy, in the setup cell). SPTLS out of the box matches it — it uses the level
to predict the change (the jet carries both), which *is* the error-correction
mechanism, **without being told the system is cointegrated or estimating a
cointegrating rank**. No win is claimed on linear data: the payoff is zero
model selection plus the white-box reading.

## 2. Nonlinear, one step — SPTLS recovers a map VAR cannot

Three quadratically-coupled series. A VAR is linear, so it cannot represent the
map at any lag; the quadratic dictionary recovers it. One-step walk-forward on
a held-out tail.

In [ ]:
def coupled(T=400, seed=2, noise=0.0):
    r = np.random.default_rng(seed); x,y,z = 0.4,0.5,0.3; out=[]
    for _ in range(300): x,y,z = 3.7*x*(1-x)-0.12*y, 3.5*y*(1-y)+0.10*z, 3.9*z*(1-z)-0.08*x
    for _ in range(T):
        x,y,z = 3.7*x*(1-x)-0.12*y, 3.5*y*(1-y)+0.10*z, 3.9*z*(1-z)-0.08*x
        out.append([x+noise*r.standard_normal(), y+noise*r.standard_normal(), z+noise*r.standard_normal()])
    return np.array(out)

Y = coupled(); st = int(0.8*len(Y))
Pv, Ps, G = [], [], []
for t in range(st, len(Y)):
    G.append(Y[t]); Pv.append(gsd.OLS(2).fit(Y[:t]).forecast_next(Y[:t]))
    Ps.append(gsd.SPTLS(dictionary="quadratic").fit(Y[:t]).forecast_next(Y[:t]))
Pv, Ps, G = np.array(Pv), np.array(Ps), np.array(G)
print("one-step walk-forward R2 per series:")
print("   VAR(2)            :", [round(gsd.metrics.r2(Pv[:,i],G[:,i]),3) for i in range(3)])
print("   SPTLS (quadratic) :", [round(gsd.metrics.r2(Ps[:,i],G[:,i]),3) for i in range(3)])

fig, ax = plt.subplots(1, 2, figsize=(8.5,3.6), sharex=True, sharey=True)
for a, P, nm in [(ax[0],Pv,"VAR(2)"), (ax[1],Ps,"SPTLS (quadratic)")]:
    a.scatter(G[:,0], P[:,0], s=10, color=("C3" if nm=="VAR(2)" else "C0"))
    lim=[G[:,0].min(),G[:,0].max()]; a.plot(lim,lim,"0.6",lw=1); a.set_title(nm); a.set_xlabel("actual")
ax[0].set_ylabel("predicted (one-step)")
fig.suptitle("Nonlinear map: predicted vs actual (on the diagonal = perfect)"); fig.tight_layout(); plt.show()

## 3. Deterministic chaos — forecasting to the Lyapunov horizon

The coupled system is chaotic. A chaotic system is *deterministic*; its natural
regime is high signal-to-noise (heavy noise would erase the attractor itself).
At realistic SNR, SPTLS recovers the map and forecasts to the system's
intrinsic predictability horizon, then degrades — the limit **any** method
faces. A linear VAR fails *structurally from step 1*, at every horizon and
every noise level.

In [ ]:
hs = [1,2,3,5,8,12]; sig = np.std(coupled(noise=0)[:,0])
for noise in (0.002, 0.005):
    HV=[]; HS=[]
    for s in range(8):
        Y=coupled(seed=s, noise=noise); tr,act=Y[:-12],Y[-12:]; sd=np.std(act,0)+1e-9
        fv, fs = fc(gsd.OLS(2),tr,12), fc(gsd.SPTLS(dictionary="quadratic"),tr,12)
        HV.append([np.mean(np.sqrt(np.mean(((fv[:hh]-act[:hh])/sd)**2,0))) for hh in hs])
        HS.append([np.mean(np.sqrt(np.mean(((fs[:hh]-act[:hh])/sd)**2,0))) for hh in hs])
    print(f"SNR~{sig/noise:.0f}  horizon {hs}")
    print("   VAR(2)    :", np.round(np.mean(HV,0),2))
    print("   SPTLS quad:", np.round(np.mean(HS,0),2))

In [ ]:
# trajectory at realistic chaotic SNR: SPTLS tracks the attractor, VAR cannot
Y = coupled(seed=0, noise=0.002); h=10; tr,act=Y[:-h],Y[-h:]
fv, fs = fc(gsd.OLS(2),tr,h), fc(gsd.SPTLS(dictionary="quadratic"),tr,h)
n=len(tr); t0=np.arange(n-25,n); t1=np.arange(n,n+h)
fig, ax = plt.subplots(figsize=(8.5,3.4))
ax.plot(t0, tr[-25:,0], color="0.6", lw=1)
ax.plot(t1, act[:,0], "o-", color="0.15", ms=4, lw=1.6, label="actual")
ax.plot(t1, fv[:,0], "--", color="C3", lw=2, label="VAR(2)")
ax.plot(t1, fs[:,0], ":", color="C0", lw=2, label="SPTLS (quadratic)")
ax.axvspan(n, n+h, color="0.93"); ax.set_title("Chaotic forecast (SNR~86): SPTLS tracks to the Lyapunov horizon, VAR fails from step 1")
ax.legend(fontsize=8); plt.tight_layout(); plt.show()

## 4. Cost — a fraction of the econometric workflow

SPTLS is **one closed-form least-squares solve** — no unit-root pre-testing,
no lag-order search, no cointegration-rank test, no differencing decision. The
fitting call alone is already cheaper than just the VAR's lag search; against
the *full* workflow (ADF/KPSS + Johansen + lag/variant selection) the gap is an
order of magnitude or more.

In [ ]:
import time
Yt = coint(T=400); N = 200
t0=time.perf_counter()
for _ in range(N): gsd.SPTLS().fit(Yt)
ms_sptls=(time.perf_counter()-t0)/N*1000
t0=time.perf_counter()
for _ in range(N): gsd.OLS(aic_p(Yt)).fit(Yt)           # VAR + AIC lag search only
ms_var=(time.perf_counter()-t0)/N*1000
print(f"per-fit: SPTLS one-shot {ms_sptls:.2f} ms  |  VAR + AIC lag search {ms_var:.2f} ms"
      f"  ({ms_var/ms_sptls:.0f}x), before any unit-root / cointegration testing")

## Takeaway — SPTLS is a natural extension of OLS, and inherits its virtues

Everything above follows from one fact: **SPTLS is ordinary least squares,
moved from lagged values to the kinematic jet $q=(z,\Delta z,\Delta^2 z)$.** A
single closed-form solve — so it keeps OLS's speed, its closed-form
estimator, and its transparency (every coefficient is readable) — while the
jet lets that same solve span behaviours a lagged-value OLS/VAR cannot.

With no model-selection ritual, SPTLS:

- **ties** a tuned VAR (stationary) and a Johansen VECM (cointegrated) —
  reproducing the error-correction without estimating a cointegrating rank;
- **recovers** a nonlinear map one-step where a linear model structurally
  cannot (`dictionary="quadratic"`);
- **forecasts deterministic chaos to the predictability horizon**, while a
  linear VAR fails from step 1 (only the chaos itself eventually limits it);
- costs **a fraction** of the econometric workflow — one solve, no pre-testing.

In one line: SPTLS is OLS for *any* trajectory — same speed, same closed form,
same white-box transparency (the reading in `gsd_diagnostics.ipynb`), with the
reach extended to every regime, and **no econometric headache** of choosing
integration order, lag length, or model class first.